In [20]:
from tokenizers import Tokenizer, models, decoders, trainers
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

# Add more tokens if you want
tokens = [
    "A", "R", "N", "D", "C", "Q", "E", "G", "H", "I",
    "L", "K", "M", "F", "P", "S", "T", "W", "Y", "V",
    "[PAD]", "[start-of-document]", "[end-of-document]", 
    "[sequence-start]", "[sequence-end]", "[UNK]", "[MASK]"
]
tokenizer.add_tokens(tokens)

tokenizer.model.unk_token = "[UNK]"

trainer = trainers.BpeTrainer(special_tokens=tokens)
tokenizer.train_from_iterator([], trainer=trainer)

tokenizer.save("protfam_tokenizer.json")


In [21]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="protfam_tokenizer.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[start-of-document]",
    sep_token="[end-of-document]",
    mask_token="[MASK]" # Add mask token if needed
)

# Test the tokenizer
# example_sequence = "ARNDC [start-of-document] QEGHIL KMFPST WYV [end-of-document] [PAD] [UNK]"
# tokens = tokenizer.tokenize(example_sequence)
# ids = tokenizer.convert_tokens_to_ids(tokens)

# print("Tokens:", tokens)
# print("Token IDs:", ids)


In [24]:
sequences_example = ["ARNDC [start-of-document] QEGHIL KMFPST WYV [end-of-document] [PAD] [UNK]", "KMFPST[MASK]"]
ids = tokenizer.batch_encode_plus(sequences_example,
                                  add_special_tokens=True,
                                  padding="longest",
                                  return_tensors='pt')

In [25]:
ids

{'input_ids': tensor([[ 0,  1,  2,  3,  4, 21,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
         17, 18, 19, 22, 20, 25],
        [11, 12, 13, 14, 15, 16, 26, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
         20, 20, 20, 20, 20, 20]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

In [26]:
tokenizer.save_pretrained("protfam")

('protfam/tokenizer_config.json',
 'protfam/special_tokens_map.json',
 'protfam/tokenizer.json')

In [ ]:
from huggingface_hub import notebook_login

# Log in to the Hugging Face Hub
# Should we create a new hf repo?
notebook_login()

In [ ]:
tokenizer.push_to_hub("protfam")